# Scraping Data Ulasan Play Store (Indonesia)

Notebook ini mengambil data ulasan aplikasi Duolingo secara mandiri dari internet menggunakan `google-play-scraper`.

Target:
- Minimal 10.000 sampel
- Data disimpan ke CSV untuk dipakai pada notebook analisis sentimen

Sumber data:
- Google Play Store

In [1]:
%pip install -q google-play-scraper pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [10]:
import warnings
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from google_play_scraper import Sort, reviews

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Scrapping review aplikasi Duolingo
APP_ID = "com.levelinfinite.sgameGlobal"
APP_NAME = "Honor of King"
LANG = "id"
COUNTRY = "id"
TARGET_COUNT = 12000
MAX_BATCH = 2000

OUTPUT_DIR = Path("content")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "reviews_hok.csv"

print(f"Target total sampel: ~{TARGET_COUNT}".replace(",", "."))

Target total sampel: ~12000


In [11]:
def scrape_reviews_for_app(app_id: str, app_name: str, target_count: int = 2000):
    """Scrape review sampai target terpenuhi atau data habis."""
    all_rows = []
    token = None

    while len(all_rows) < target_count:
        batch_count = min(MAX_BATCH, target_count - len(all_rows))
        result, token = reviews(
            app_id,
            lang=LANG,
            country=COUNTRY,
            sort=Sort.MOST_RELEVANT,
            count=batch_count,
            continuation_token=token,
        )

        if not result:
            break

        for r in result:
            all_rows.append(
                {
                    "app_id": app_id,
                    "app_name": app_name,
                    "review_id": r.get("reviewId"),
                    "user_name": r.get("userName"),
                    "score": r.get("score"),
                    "content": r.get("content"),
                    "thumbs_up": r.get("thumbsUpCount"),
                    "at": r.get("at"),
                    "app_version": r.get("appVersion"),
                }
            )

        if token is None:
            break

    return all_rows

In [12]:
all_reviews = scrape_reviews_for_app(APP_ID, APP_NAME, target_count=TARGET_COUNT)

raw_df = pd.DataFrame(all_reviews)
print(f"\nTotal ulasan terkumpul (raw): {len(raw_df):,}".replace(",", "."))
raw_df.head()


Total ulasan terkumpul (raw): 12.000


,app_id,app_name,review_id,user_name,score,content,thumbs_up,at,app_version
0,com.levelinfinite.sgameGlobal,Honor of King,2c793cc6-a533-4756-899c-929fe0905f1f,Muhammad Ramadani,5,"game nya seru, developer nya baik, heronya balance daripada moba lain. hal yang aku permasalahkan; sering kali layar...",11,2026-04-03 14:50:36,11.3.1.8
1,com.levelinfinite.sgameGlobal,Honor of King,03d587fa-daa9-4a40-b7c0-31ad6207ec45,Andriyanto,2,"tolong perbaiki peforma peningkatan sinyal nya, kadang yang diutamakan itu pengalaman bermain, tolong perbaiki serve...",2,2026-04-23 17:57:13,11.3.1.9
2,com.levelinfinite.sgameGlobal,Honor of King,f9f2926a-cfd2-4c77-b668-48813c845d28,Andika,2,"gamenya udah bagus cuma ada kekurangan di fitur download, yaitu skin dalam match (yang dimiliki) sama yang (belum di...",1,2026-04-25 07:25:37,11.3.1.9
3,com.levelinfinite.sgameGlobal,Honor of King,6febf65b-1bcd-4893-879a-03b7679ea16c,icooo,1,"game rusak game rusak, lain kali kalo update jangan cuma mikirin skin,collab,dll. pentingin juga optimalisasi jaring...",0,2026-04-27 08:59:50,11.3.1.9
4,com.levelinfinite.sgameGlobal,Honor of King,7a0f84f8-5c43-497a-b6c2-35b045d29061,Nur Juminati,5,"Suka banget sama game nya, soalnya hero nya juga kebanyakan cakep-cakep:b. Cuma ya.. Kekurangan nya itu cuma satu, j...",1,2026-04-24 00:44:56,11.3.1.9


In [13]:
df = raw_df.copy()

print("Distribusi skor:")
display(df["score"].value_counts().sort_index())

# Simpan ke CSV
ordered_cols = [
    "review_id", "app_id", "app_name", "user_name", "score", "content", "thumbs_up", "at", "app_version"
]
existing_cols = [c for c in ordered_cols if c in df.columns]
df[existing_cols].to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"\nFile CSV tersimpan di: {OUTPUT_CSV.resolve()}")
print(f"Jumlah sampel akhir: {len(df):,}".replace(",", "."))

Distribusi skor:


score
1    3948
2    1180
3    1370
4    1516
5    3986
Name: count, dtype: int64


File CSV tersimpan di: C:\Users\DianErdiana\dianerdiana-learning\machine-learning-dicoding\submission-deep-learning-01\content\reviews_hok.csv
Jumlah sampel akhir: 12.000


In [8]:
df.sample(5, random_state=42)[["app_name", "score", "content"]]

,app_name,score,content
1935,Honor of King,5,"Ternyata emang di daerah tempat tinggal saya tidak bisa main HOK dengan lancar, jaringan internet saya bagus, yang l..."
6494,Honor of King,1,"HP saya suport game ini namun tidak tau kenapa layarnya tiba-tiba freeze, dan banyak lag terjadi padahal di game seb..."
1720,Honor of King,1,"saya udah topup banyak tapi grafiknya tiba tiba gak bisa di atur ke lebih tinggi, percuma punya skin bagus tapi effe..."
9120,Honor of King,4,Banyak player gblk di game ini terutama tier DM-MASTER banyak player gak paham cara main seperti timeing posisi dll ...
360,Honor of King,3,game nya udh bagus... kalo efek animasi luarnya udh mantap... cuman dari segi in game nya itu masih kurang smoot dar...
